# **Building and Training a Feedforward Neural Network for Language Modeling**

This project explores the use of Feedforward Neural Networks (FNNs) in language modeling. The primary objective is to build a neural network that learns word relationships and generates meaningful text sequences. The implementation is done using PyTorch, covering key aspects of Natural Language Processing (NLP), such as:
* Tokenization & Indexing: Converting text into numerical representations.
* Embedding Layers: Mapping words to dense vector representations for efficient learning.
* Context-Target Pair Generation (N-grams): Structuring training data for sequence prediction.
* Multi-Class Neural Network: Designing a model to predict the next word in a sequence.

The training process includes optimizing the model with loss functions and backpropagation techniques to improve accuracy and coherence in text generation. By the end of the project, you will have a working FNN-based language model capable of generating text sequences.



# **Objectives**

After completing this lab, you will be able to:

 - Implement a feedforward neural network using the PyTorch framework, including embedding layers, for language modeling tasks.
 - Fine-tune the output layer of the neural network for optimal performance in text generation.
 - Apply various training strategies and fundamental Natural Language Processing (NLP) techniques, such as tokenization and sequence analysis, to improve text generation.


#### Import Libraries

In [7]:
from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import nltk
import torch
import re
import string
from nltk.tokenize import word_tokenize


## Feedforward Neural Networks (FNNs) for language models

FNNs, or Multi-Layer Perceptrons, serve as the foundational components for comprehending neural networks in natural language processing (NLP). In NLP tasks, FNNs process textual data by transforming it into numerical vectors known as embeddings. Subsequently, these embeddings are input to the network to predict language facets, such as the upcoming word in a sentence or the sentiment of a text.

Let's consider the following song lyrics for our analysis.


In [22]:
song= """We are no strangers to love
You know the rules and so do I
A full commitments what Im thinking of
You wouldnt get this from any other guy
I just wanna tell you how Im feeling
Gotta make you understand
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you
Weve known each other for so long
Your hearts been aching but youre too shy to say it
Inside we both know whats been going on
We know the game and were gonna play it
And if you ask me how Im feeling
Dont tell me youre too blind to see
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you
Weve known each other for so long
Your hearts been aching but youre too shy to say it
Inside we both know whats been going on
We know the game and were gonna play it
I just wanna tell you how Im feeling
Gotta make you understand
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you"""

### Data Preprocessing

In [6]:
def preprocess_string(s):
    
    # Remove all non-word characters (everything except letters and numbers)
    # \w matches any word character (letters, numbers, and underscores)
    # \s matches any whitespace characters
    # ^ inside [] negates the selection, so [^\w\s] matches anything that's NOT a word character or whitespace.
    s = re.sub(r"[^\w\s]",'', s)
    
    # Remove all whitespace characters (spaces, tabs, newlines)
    # \s+ matches one or more whitespace characters.
    s = re.sub(r"[\s+]",'', s)
    
    # Remove all digits (0-9)
    # \d matches any digit character.
    s = re.sub(r"[\d]",'',s)
    
    return s

In [14]:
def preprocessing(words):
    
    tokens = word_tokenize(words)
    tokens_preprocessed = [preprocess_string(token) for token in tokens]
    
    return [word.lower() for word in tokens_preprocessed if len(word) !=0 and
            (word not in string.punctuation)]
        

In [ ]:
## tokenized the song
tokens = preprocessing(song)
tokens

['we',
 'are',
 'no',
 'strangers',
 'to',
 'love',
 'you',
 'know',
 'the',
 'rules',
 'and',
 'so',
 'do',
 'i',
 'a',
 'full',
 'commitments',
 'what',
 'im',
 'thinking',
 'of',
 'you',
 'wouldnt',
 'get',
 'this',
 'from',
 'any',
 'other',
 'guy',
 'i',
 'just',
 'wan',
 'na',
 'tell',
 'you',
 'how',
 'im',
 'feeling',
 'got',
 'ta',
 'make',
 'you',
 'understand',
 'never',
 'gon',
 'na',
 'give',
 'you',
 'up',
 'never',
 'gon',
 'na',
 'let',
 'you',
 'down',
 'never',
 'gon',
 'na',
 'run',
 'around',
 'and',
 'desert',
 'you',
 'never',
 'gon',
 'na',
 'make',
 'you',
 'cry',
 'never',
 'gon',
 'na',
 'say',
 'goodbye',
 'never',
 'gon',
 'na',
 'tell',
 'a',
 'lie',
 'and',
 'hurt',
 'you',
 'weve',
 'known',
 'each',
 'other',
 'for',
 'so',
 'long',
 'your',
 'hearts',
 'been',
 'aching',
 'but',
 'youre',
 'too',
 'shy',
 'to',
 'say',
 'it',
 'inside',
 'we',
 'both',
 'know',
 'whats',
 'been',
 'going',
 'on',
 'we',
 'know',
 'the',
 'game',
 'and',
 'were',
 'gon',

### Vocabulary 

In [23]:
vocab = set(tokens)
vocab

{'a',
 'aching',
 'and',
 'any',
 'are',
 'around',
 'ask',
 'been',
 'blind',
 'both',
 'but',
 'commitments',
 'cry',
 'desert',
 'do',
 'dont',
 'down',
 'each',
 'feeling',
 'for',
 'from',
 'full',
 'game',
 'get',
 'give',
 'going',
 'gon',
 'goodbye',
 'got',
 'guy',
 'hearts',
 'how',
 'hurt',
 'i',
 'if',
 'im',
 'inside',
 'it',
 'just',
 'know',
 'known',
 'let',
 'lie',
 'long',
 'love',
 'make',
 'me',
 'na',
 'never',
 'no',
 'of',
 'on',
 'other',
 'play',
 'rules',
 'run',
 'say',
 'see',
 'shy',
 'so',
 'strangers',
 'ta',
 'tell',
 'the',
 'thinking',
 'this',
 'to',
 'too',
 'understand',
 'up',
 'wan',
 'we',
 'were',
 'weve',
 'what',
 'whats',
 'wouldnt',
 'you',
 'your',
 'youre'}

In [31]:
text_pipeline_old = lambda x: vocab(word_tokenize(x))
text_pipeline_old(song)[0:10]


TypeError: 'set' object is not callable

**For this error we can callable sets, because vocab is created using sets instead of build_vocab so we have to define get_tokenizer uisng torchtext**

In [32]:
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

In [59]:
song= """We are no strangers to love
You know the rules and so do I
A full commitments what Im thinking of
You wouldnt get this from any other guy
I just wanna tell you how Im feeling
Gotta make you understand
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you
Weve known each other for so long
Your hearts been aching but youre too shy to say it
Inside we both know whats been going on
We know the game and were gonna play it
And if you ask me how Im feeling
Dont tell me youre too blind to see
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you
Weve known each other for so long
Your hearts been aching but youre too shy to say it
Inside we both know whats been going on
We know the game and were gonna play it
I just wanna tell you how Im feeling
Gotta make you understand
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you
Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you"""

In [62]:
tokenizer = get_tokenizer("basic_english")
        
vocabulary = build_vocab_from_iterator(map(tokenizer,song.split()),specials=["<unk>"])
vocabulary.set_default_index(vocabulary["<unk>"])

In [64]:
print(vocabulary(tokens)[0:10])

[21, 58, 70, 74, 25, 69, 2, 20, 31, 72]


In [65]:
# set vocabulary index (that converts raw text into indexes)
text_pipeline = lambda x: vocabulary(tokenizer(x))
text_pipeline(song)[0:10]

[21, 58, 70, 74, 25, 69, 2, 20, 31, 72]

In [67]:
index_to_token = vocabulary.get_itos()
index_to_token[58]

'are'

## Embdedding Layer

An embedding layer is a crucial element in natural language processing (NLP) and neural networks designed for sequential data. It serves to convert categorical variables, like words or discrete indexes representing tokens, into continuous vectors. This transformation facilitates training and enables the network to learn meaningful relationships among words.

Let’s consider a different vocabulary:

Vocabulary: {cat, dog, bird, fish}

Each word is assigned a unique index:

Indices: {0, 1, 2, 3}

When using an embedding layer, each index is mapped to a continuous vector (randomly initialized at first):

Vector for index 0 (cat): [0.5, -0.2, 0.1]
Vector for index 1 (dog): [-0.3, 0.9, 0.4]
Vector for index 2 (bird): [0.7, 0.3, -0.6]
Vector for index 3 (fish): [-0.1, 0.2, 0.8]

In [68]:
import torch.nn as nn 

In [70]:

def gen_embeddeing(vocab):
    
    # define number of dimesions
    embeded_dim = 20
    # define vocabulary size
    vocab_size = len(vocab)
    
    # define an embedding layer
    embedding = nn.Embedding(vocab_size, embeded_dim)
    
    return embedding
    


**Embeddings**: Obtain the embedding for the first word with index 0 or 1. Don't forget that you have to convert the input into a tensor. The embeddings are initially initialized randomly, but as the model undergoes training, words with similar meanings gradually come to cluster closer together


In [72]:
embeddings = gen_embeddeing(vocabulary)
embeddings

Embedding(79, 20)

In [73]:
for n in range(2): 
    embedding=embeddings(torch.tensor(n))
    print("word",index_to_token[n])
    print("index",n)
    print( "embedding", embedding)
    print("embedding shape", embedding.shape)

word <unk>
index 0
embedding tensor([ 0.3665, -1.6327,  0.2447, -0.0286,  0.1160, -0.1051, -1.1051,  1.0554,
         0.2749,  0.6104, -1.2929, -0.4480, -1.2311, -1.0799,  1.2675, -0.9319,
         0.8331, -1.1249,  0.1851,  1.3899], grad_fn=<EmbeddingBackward0>)
embedding shape torch.Size([20])
word gonna
index 1
embedding tensor([-0.3864,  0.8002,  0.4039, -0.7351,  0.4325,  0.0064, -1.5167,  2.3986,
         0.8679,  1.5054, -1.5452,  0.1719,  1.3504, -0.4959, -1.6823, -0.6330,
        -0.5305, -0.1385, -1.8068, -0.7959], grad_fn=<EmbeddingBackward0>)
embedding shape torch.Size([20])
